# TensorBoard Loss Landscape Plugin Demo

This notebook demonstrates how to combine PySlice loss landscape analysis with TensorBoard for improved neural network training insights. We'll train a small network on the Boston Housing dataset and visualize:

1. **Axis-aligned slicing** - Parameter sensitivity analysis
2. **Linear interpolation** - Training path visualization  
3. **Random direction slicing** - Local loss landscape topology

## Use Cases Demonstrated:
- Detecting sharp vs flat minima for generalization insights
- Diagnosing training dynamics and saddle points
- Comparing checkpoints for model selection
- Identifying sensitive parameters for regularization
- Validating learning rate schedules
- Early stopping based on loss landscape geometry

## Task 1: Data Setup, Model Creation, and Training Function

In [9]:
# Standard libraries
import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# PySlice components
from pysclice.slicers import LinearInterpolationSlicer, AxisParallelSlicer, RandomDirectionSlicer
from pysclice.core import ModelWrapper

# TensorBoard plugin components
import sys
sys.path.append('../../tensorboard_plugin')  # Add path to tensorboard plugin
from tensorboard_loss_slicer.summary_v2 import log_slice
import tensorflow as tf

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Libraries loaded successfully")

Libraries loaded successfully


In [10]:
# Load Boston Housing dataset
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
data = pd.read_csv('./data/housing.csv', header=None, delimiter=r"\s+", names=column_names)

# Remove outliers (MEDV >= 50.0)
data = data[~(data['MEDV'] >= 50.0)]

# Select features with good correlation to target
feature_cols = ['LSTAT', 'INDUS', 'NOX', 'PTRATIO', 'RM', 'TAX', 'DIS', 'AGE']
X = data[feature_cols].values
y = data['MEDV'].values

# Apply log transformation to reduce skewness
y = np.log1p(y)

# Scale features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Dataset loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")
print(f"Features: {X_train.shape[1]}, Target range: {y_train.min():.3f} - {y_train.max():.3f}")

Dataset loaded: 392 train, 98 test samples
Features: 8, Target range: 1.792 - 3.908


In [11]:
# Define small neural network for regression
class BostonHousingNet(nn.Module):
    """Small network for Boston Housing regression - easy to analyze"""
    
    def __init__(self, input_size=8, hidden_size=16, output_size=1):
        super(BostonHousingNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size // 2, output_size)
        
    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

# Model configuration
INPUT_SIZE = X_train.shape[1]
HIDDEN_SIZE = 16
OUTPUT_SIZE = 1

# Create model
model = BostonHousingNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {total_params} parameters")
print(f"Architecture: {INPUT_SIZE} -> {HIDDEN_SIZE} -> {HIDDEN_SIZE//2} -> {OUTPUT_SIZE}")

Model created with 289 parameters
Architecture: 8 -> 16 -> 8 -> 1


In [12]:
# Training and evaluation functions
def evaluate_model(model, data_loader, criterion):
    """Evaluate model on given dataset"""
    model.eval()
    total_loss = 0
    total_samples = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)
            total_samples += inputs.size(0)
    
    return total_loss / total_samples

def compute_gradient_norm(model):
    """Compute L2 norm of gradients"""
    total_norm = 0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    return total_norm ** 0.5

def train_epoch(model, train_loader, optimizer, criterion, writer, epoch):
    """Train one epoch and log metrics"""
    model.train()
    total_loss = 0
    total_samples = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        
        # Compute gradient norm before optimizer step
        grad_norm = compute_gradient_norm(model)
        
        optimizer.step()
        
        total_loss += loss.item() * inputs.size(0)
        total_samples += inputs.size(0)
        
        # Log batch-level metrics
        global_step = epoch * len(train_loader) + batch_idx
        writer.add_scalar('Train/BatchLoss', loss.item(), global_step)
        writer.add_scalar('Train/GradNorm', grad_norm, global_step)
    
    return total_loss / total_samples

print("Training functions defined")

Training functions defined


## Task 2: Training with Checkpoint Snapshots

Now we'll train the model and save snapshots at key points for loss landscape analysis.

In [13]:
# Training configuration
EPOCHS = 50
LEARNING_RATE = 0.01
LOG_DIR = './tensorboard_logs/boston_housing'

# Create fresh model and optimizer
model = BostonHousingNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.9)

# TensorBoard writer
os.makedirs(LOG_DIR, exist_ok=True)
writer = SummaryWriter(LOG_DIR)

# Storage for checkpoints and metrics
checkpoints = {}
slice_epochs = [0, 10, 25, 49]  # Epochs to perform slicing analysis

print(f"Starting training for {EPOCHS} epochs")
print(f"Logging to: {LOG_DIR}")
print(f"Will perform slicing analysis at epochs: {slice_epochs}")

Starting training for 50 epochs
Logging to: ./tensorboard_logs/boston_housing
Will perform slicing analysis at epochs: [0, 10, 25, 49]


In [14]:
# Main training loop
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(EPOCHS):
    # Train one epoch
    train_loss = train_epoch(model, train_loader, optimizer, criterion, writer, epoch)
    
    # Evaluate on validation set
    val_loss = evaluate_model(model, test_loader, criterion)
    
    # Log epoch-level metrics
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('Learning/LR', optimizer.param_groups[0]['lr'], epoch)
    
    # Track best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        checkpoints['best'] = copy.deepcopy(model.state_dict())
    
    # Save snapshots at key epochs
    if epoch in slice_epochs:
        checkpoints[f'epoch_{epoch}'] = copy.deepcopy(model.state_dict())
        print(f"Epoch {epoch}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")
    
    # Save initial and final states
    if epoch == 0:
        checkpoints['initial'] = copy.deepcopy(model.state_dict())
    elif epoch == EPOCHS - 1:
        checkpoints['final'] = copy.deepcopy(model.state_dict())

writer.close()
print(f"Training completed. Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}")
print(f"Saved {len(checkpoints)} checkpoints for analysis")

Epoch 0: Train Loss = 4.3837, Val Loss = 0.1250


Epoch 10: Train Loss = 0.1313, Val Loss = 0.1234
Epoch 25: Train Loss = 0.1318, Val Loss = 0.1256
Epoch 25: Train Loss = 0.1318, Val Loss = 0.1256
Epoch 49: Train Loss = 0.1319, Val Loss = 0.1271
Training completed. Best validation loss: 0.1192 at epoch 2
Saved 7 checkpoints for analysis
Epoch 49: Train Loss = 0.1319, Val Loss = 0.1271
Training completed. Best validation loss: 0.1192 at epoch 2
Saved 7 checkpoints for analysis


## Task 3: TensorBoard Loss Landscape Logging

Now we'll use the custom TensorBoard plugin to log loss landscape slices for analysis.

In [15]:
# Prepare slice analysis data (smaller subset for faster slicing)
slice_size = 100
slice_indices = np.random.choice(len(X_train), slice_size, replace=False)
slice_X = torch.FloatTensor(X_train[slice_indices])
slice_y = torch.FloatTensor(y_train[slice_indices]).unsqueeze(1)

# Create model wrapper for slicing
def create_model_wrapper(state_dict):
    """Create ModelWrapper for slicing analysis"""
    temp_model = BostonHousingNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
    temp_model.load_state_dict(state_dict)
    return ModelWrapper(temp_model, criterion, (slice_X, slice_y))

# Analysis configuration
SLICE_SAMPLES = 30  # Resolution for slicing
SLICE_RANGE = 0.5   # Range for parameter exploration

print(f"Slice analysis data prepared: {slice_size} samples")
print(f"Slice resolution: {SLICE_SAMPLES} samples per slice")
print(f"Parameter exploration range: ±{SLICE_RANGE}")

Slice analysis data prepared: 100 samples
Slice resolution: 30 samples per slice
Parameter exploration range: ±0.5


In [16]:
# Axis-Parallel Slicing Analysis
print("Performing axis-parallel slicing analysis...")

# Create TensorFlow writer for the plugin
tf_writer = tf.summary.create_file_writer(LOG_DIR)

for checkpoint_name, state_dict in checkpoints.items():
    if checkpoint_name in ['initial', 'epoch_10', 'best', 'final']:
        print(f"Analyzing {checkpoint_name}...")
        
        # Create model wrapper
        wrapper = create_model_wrapper(state_dict)
        
        # Axis-parallel slicing
        axis_slicer = AxisParallelSlicer(wrapper)
        axis_data = axis_slicer.sample_focus_points_and_slice(
            center_point=None,
            n_points=5,  # Focus points around current model
            sampling_method="lhs",
            radius=0.1,
            bounds=(-SLICE_RANGE, SLICE_RANGE),
            n_samples_per_slice=SLICE_SAMPLES,
            bounds_mode="relative",
            seed=42
        )
        
        # Log to TensorBoard using unified log_slice function
        step = int(checkpoint_name.split('_')[-1]) if 'epoch' in checkpoint_name else 0
        if checkpoint_name == 'best':
            step = best_epoch
        elif checkpoint_name == 'final':
            step = EPOCHS - 1
        
        with tf_writer.as_default():
            log_slice(
                name=f"AxisParallel/{checkpoint_name}",
                slice_data=axis_data,
                step=step,
                description=f"Parameter sensitivity analysis at {checkpoint_name}"
            )

print("Axis-parallel slicing complete")

Performing axis-parallel slicing analysis...
Analyzing best...
Analyzing best...
Analyzing initial...
Analyzing initial...
Analyzing epoch_10...
Analyzing final...
Axis-parallel slicing complete


In [17]:
# Linear Interpolation Slicing Analysis
print("Performing linear interpolation analysis...")

# Create wrappers for key checkpoints
wrapper_initial = create_model_wrapper(checkpoints['initial'])
wrapper_best = create_model_wrapper(checkpoints['best'])
wrapper_final = create_model_wrapper(checkpoints['final'])

# Get parameter points
point_initial = wrapper_initial.get_parameters()
point_best = wrapper_best.get_parameters()
point_final = wrapper_final.get_parameters()

# Linear interpolation between key points
linear_slicer = LinearInterpolationSlicer(wrapper_final)

# Initial → Best
linear_data_init_best = linear_slicer.slice(
    start_point=point_initial,
    end_point=point_best,
    n_samples=SLICE_SAMPLES
)

with tf_writer.as_default():
    log_slice(
        name="LinearInterp/InitialToBest",
        slice_data=linear_data_init_best,
        step=best_epoch,
        description="Training path from initialization to best model"
    )

# Best → Final  
linear_data_best_final = linear_slicer.slice(
    start_point=point_best,
    end_point=point_final,
    n_samples=SLICE_SAMPLES
)

with tf_writer.as_default():
    log_slice(
        name="LinearInterp/BestToFinal", 
        slice_data=linear_data_best_final,
        step=EPOCHS-1,
        description="Path from best model to final (potential overfitting)"
    )

# Initial → Final (direct path)
linear_data_init_final = linear_slicer.slice(
    start_point=point_initial,
    end_point=point_final,
    n_samples=SLICE_SAMPLES
)

with tf_writer.as_default():
    log_slice(
        name="LinearInterp/InitialToFinal",
        slice_data=linear_data_init_final,
        step=EPOCHS-1,
        description="Direct path from initialization to final model"
    )

print("Linear interpolation analysis complete")

Performing linear interpolation analysis...
Linear interpolation analysis complete


In [18]:
# Random Direction Slicing Analysis  
print("Performing random direction (2D landscape) analysis...")

# Analyze 2D loss landscapes around key checkpoints
for checkpoint_name, state_dict in [('initial', checkpoints['initial']), 
                                   ('best', checkpoints['best']), 
                                   ('final', checkpoints['final'])]:
    
    print(f"2D landscape analysis for {checkpoint_name}...")
    
    wrapper = create_model_wrapper(state_dict)
    random_slicer = RandomDirectionSlicer(wrapper)
    
    # 2D random direction slice
    random_data = random_slicer.slice(
        center_point=None,
        n_samples=SLICE_SAMPLES,
        x_range=(-SLICE_RANGE, SLICE_RANGE),
        y_range=(-SLICE_RANGE, SLICE_RANGE),
        normalize_directions=True,
        ensure_orthogonal=True
    )
    
    # Determine step for logging
    step = 0
    if checkpoint_name == 'best':
        step = best_epoch
    elif checkpoint_name == 'final':
        step = EPOCHS - 1
    
    with tf_writer.as_default():
        log_slice(
            name=f"RandomDirection/{checkpoint_name}",
            slice_data=random_data,
            step=step,
            description=f"2D loss landscape topology around {checkpoint_name} model"
        )

print("Random direction analysis complete")

Performing random direction (2D landscape) analysis...
2D landscape analysis for initial...
2D landscape analysis for best...
2D landscape analysis for final...
Random direction analysis complete
2D landscape analysis for final...
Random direction analysis complete


In [19]:
# Close writer and provide analysis summary
tf_writer.close()

print("=" * 60)
print("TENSORBOARD LOSS LANDSCAPE ANALYSIS COMPLETE")
print("=" * 60)

print(f"\nTraining Summary:")
print(f"- Total epochs: {EPOCHS}")
print(f"- Best validation loss: {best_val_loss:.4f} at epoch {best_epoch}")
print(f"- Model parameters: {total_params}")

print(f"\nSlicing Analysis Logged:")
print(f"- Axis-parallel slices: 4 checkpoints")
print(f"- Linear interpolations: 3 paths") 
print(f"- 2D random direction slices: 3 checkpoints")

print(f"\nTensorBoard Logs:")
print(f"- Log directory: {LOG_DIR}")
print(f"- View with: tensorboard --logdir {LOG_DIR}")

print(f"\nAnalysis Use Cases Demonstrated:")
print(f"- Parameter sensitivity evolution (axis-parallel)")
print(f"- Training path smoothness (linear interpolation)")
print(f"- Sharp vs flat minima detection (2D landscapes)")
print(f"- Checkpoint comparison for model selection")
print(f"- Overfitting detection (best vs final comparison)")

print(f"\nInterpretation Guide:")
print(f"- Flat landscapes → better generalization")
print(f"- Sharp minima → higher overfitting risk") 
print(f"- Smooth training paths → stable optimization")
print(f"- High parameter sensitivity → regularization targets")
print(f"- Loss barriers between checkpoints → mode connectivity insights")

TENSORBOARD LOSS LANDSCAPE ANALYSIS COMPLETE

Training Summary:
- Total epochs: 50
- Best validation loss: 0.1192 at epoch 2
- Model parameters: 289

Slicing Analysis Logged:
- Axis-parallel slices: 4 checkpoints
- Linear interpolations: 3 paths
- 2D random direction slices: 3 checkpoints

TensorBoard Logs:
- Log directory: ./tensorboard_logs/boston_housing
- View with: tensorboard --logdir ./tensorboard_logs/boston_housing

Analysis Use Cases Demonstrated:
- Parameter sensitivity evolution (axis-parallel)
- Training path smoothness (linear interpolation)
- Sharp vs flat minima detection (2D landscapes)
- Checkpoint comparison for model selection
- Overfitting detection (best vs final comparison)

Interpretation Guide:
- Flat landscapes → better generalization
- Sharp minima → higher overfitting risk
- Smooth training paths → stable optimization
- High parameter sensitivity → regularization targets
- Loss barriers between checkpoints → mode connectivity insights


## Advanced Analysis Examples (Optional)

The following cells demonstrate additional use cases for combining loss landscape analysis with standard TensorBoard metrics.

In [20]:
# Advanced Use Case 1: Learning Rate Sensitivity Analysis
print("Analyzing learning rate sensitivity...")

# Train models with different learning rates for comparison
lr_values = [0.001, 0.01, 0.1]
lr_models = {}

for lr in lr_values:
    print(f"Training with LR={lr}...")
    
    # Create fresh model
    lr_model = BostonHousingNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
    lr_optimizer = optim.SGD(lr_model.parameters(), lr=lr, momentum=0.9)
    
    # Quick training (fewer epochs for demo)
    for epoch in range(20):
        train_loss = train_epoch(lr_model, train_loader, lr_optimizer, criterion, 
                               SummaryWriter(f"{LOG_DIR}/lr_{lr}"), epoch)
    
    # Store final model
    lr_models[lr] = copy.deepcopy(lr_model.state_dict())

# Compare loss landscapes of different LR models
tf_writer = tf.summary.create_file_writer(LOG_DIR)

for lr, state_dict in lr_models.items():
    wrapper = create_model_wrapper(state_dict)
    
    # 2D landscape around final model
    random_slicer = RandomDirectionSlicer(wrapper)
    random_data = random_slicer.slice(
        center_point=None,
        n_samples=25,
        x_range=(-0.3, 0.3),
        y_range=(-0.3, 0.3),
        normalize_directions=True,
        ensure_orthogonal=True
    )
    
    with tf_writer.as_default():
        log_slice(
            name=f"LearningRate/LR_{lr}",
            slice_data=random_data,
            step=int(lr * 1000),  # Use LR as step for comparison
            description=f"Final loss landscape with learning rate {lr}"
        )

tf_writer.close()
print("Learning rate sensitivity analysis complete")

Analyzing learning rate sensitivity...
Training with LR=0.001...
Training with LR=0.01...Training with LR=0.01...
Training with LR=0.1...

Training with LR=0.1...
Learning rate sensitivity analysis complete
Learning rate sensitivity analysis complete


In [21]:
# Advanced Use Case 2: Layer-wise Parameter Sensitivity
print("Analyzing layer-wise parameter sensitivity...")

# Custom slicer for layer-specific analysis
class LayerWiseAxisSlicer:
    def __init__(self, model_wrapper):
        self.model_wrapper = model_wrapper
        
    def slice_layer(self, layer_name, n_samples=20, param_range=0.2):
        """Slice parameters within a specific layer"""
        model = self.model_wrapper.model
        original_params = self.model_wrapper.get_parameters()
        
        # Find parameters belonging to the specified layer
        layer_indices = []
        param_idx = 0
        
        for name, param in model.named_parameters():
            if layer_name in name:
                layer_indices.extend(range(param_idx, param_idx + param.numel()))
            param_idx += param.numel()
        
        if not layer_indices:
            return None
            
        # Slice only the layer parameters
        losses = []
        perturbations = np.linspace(-param_range, param_range, n_samples)
        
        for perturbation in perturbations:
            # Create perturbed parameters
            perturbed_params = original_params.copy()
            for idx in layer_indices:
                perturbed_params[idx] += perturbation
            
            # Set model parameters and compute loss
            self.model_wrapper.set_parameters(perturbed_params)
            loss = self.model_wrapper.compute_loss()
            losses.append(loss)
        
        # Restore original parameters
        self.model_wrapper.set_parameters(original_params)
        
        return {
            'perturbations': perturbations,
            'losses': losses,
            'layer_name': layer_name,
            'n_params': len(layer_indices)
        }

# Analyze each layer's sensitivity for the best model
wrapper_best = create_model_wrapper(checkpoints['best'])
layer_slicer = LayerWiseAxisSlicer(wrapper_best)

tf_writer = tf.summary.create_file_writer(LOG_DIR)

layers_to_analyze = ['fc1', 'fc2', 'fc3']
for layer_name in layers_to_analyze:
    print(f"Analyzing layer: {layer_name}")
    
    layer_data = layer_slicer.slice_layer(layer_name, n_samples=30, param_range=0.1)
    
    if layer_data:
        # Log layer sensitivity using the unified log_slice function
        # Note: This would need a custom slice type or we can format it as linear interpolation
        formatted_data = {
            'alphas': layer_data['perturbations'],
            'losses': layer_data['losses'],
            'parameters': [],  # Not applicable for this analysis
            'type': 'linear_interpolation'
        }
        
        with tf_writer.as_default():
            log_slice(
                name=f"LayerSensitivity/{layer_name}",
                slice_data=formatted_data,
                step=best_epoch,
                description=f"Parameter sensitivity in {layer_name} ({layer_data['n_params']} parameters)"
            )

tf_writer.close()
print("Layer-wise sensitivity analysis complete")

Analyzing layer-wise parameter sensitivity...
Analyzing layer: fc1
Analyzing layer: fc2
Analyzing layer: fc3
Layer-wise sensitivity analysis complete
